# Vue2 + Vue3 框架 · 第 04 天

## 组件通信与组件进阶用法

> 本篇依据第 04 天 MD 笔记、PPT 和随堂代码整理。当天重点是 Vue 组件之间如何传递数据，以及如何封装可复用的表单、弹框和图表组件。

前几天已经学会了组件注册和单文件组件。本天要建立一套完整的组件通信思路：

- 父组件向子组件传数据：`props`。
- 子组件通知父组件更新：`$emit`。
- 非父子组件通信：事件总线、`provide/inject`。
- 组件双向绑定：`v-model`、`.sync`。
- 需要访问 DOM 或子组件实例：`ref`、`$refs`。
- 修改数据后马上操作新 DOM：`$nextTick`。

## 一、学习目标与知识地图

| 模块 | 关键知识 | 使用场景 |
| --- | --- | --- |
| 样式隔离 | `<style scoped>` | 防止组件样式相互污染 |
| 组件状态 | `data` 必须是函数 | 保证每个实例拥有独立数据 |
| 父子通信 | `props`、`$emit` | 父传子、子通知父 |
| Props 校验 | `type`、`required`、`default`、`validator` | 让组件输入更可靠 |
| 综合案例 | 组件版小黑记事本 | 列表、添加、删除、统计、持久化 |
| 非父子通信 | EventBus、`provide/inject` | 兄弟组件或跨层级传值 |
| 双向绑定 | `v-model`、`.sync` | 封装表单和弹框组件 |
| 实例访问 | `ref`、`$refs`、`$nextTick` | 图表、聚焦、测量 DOM |

### 组件通信先问两个问题

1. 数据的所有者是谁？遵循“**谁的数据谁负责**”。
2. 两个组件之间是什么关系？优先使用最简单、最明确的通信方式。

## 二、`scoped` 解决样式冲突

### 2.1 默认样式的作用范围

在单文件组件中，普通 `<style>` 里的 CSS 默认会被打包成全局样式。同名选择器可能影响其他组件：

```vue
<!-- BaseOne.vue -->
<style>
.base-one { color: red; }
</style>

<!-- BaseTwo.vue -->
<style>
.base-one { color: blue; }
</style>
```

两个组件同时使用时，样式可能互相覆盖。

### 2.2 使用 `scoped`

```vue
<style scoped>
.base-one {
  color: red;
}
</style>
```

Vue 编译后会给当前组件的元素添加类似 `data-v-abc123` 的属性，同时给选择器加上对应的属性选择器。最终只有当前组件的元素会匹配这段 CSS。

![scoped 编译示意](assets/day04/1682306517375.png)

### 2.3 使用建议

- 页面或组件专用样式优先使用 `scoped`。
- 全局重置、主题变量、公共工具类放到全局 CSS。
- `scoped` 不能阻止通过内联样式、深度选择器或全局类名造成的影响；第三方组件样式需要按组件库规则覆盖。

## 三、组件的 `data` 必须是函数

### 3.1 为什么组件中的 `data` 是函数

一个组件可能被复用很多次。如果 `data` 直接写成对象，多个实例会共享同一个对象，修改一个实例会影响其他实例。

函数写法会在每创建一个组件实例时执行一次，返回一份新的数据对象：

```js
export default {
  data () {
    return {
      count: 100
    }
  }
}
```

根实例 `new Vue({ data: {} })` 可以写对象；普通组件的 `data` 必须写函数。

![组件实例独立数据](assets/day04/1682306952078.png)

### 3.2 计数器示例

```vue
<!-- BaseCount.vue -->
<template>
  <div class="base-count">
    <button @click="count--">-</button>
    <span>{{ count }}</span>
    <button @click="count++">+</button>
  </div>
</template>

<script>
export default {
  data () {
    return { count: 100 }
  }
}
</script>
```

在页面中使用两次 `<BaseCount />`，两个计数器的值应该互不影响。

## 四、父子组件通信

组件自身的数据默认是隔离的。父子关系下最常用的通信方向是：

```text
父组件 --props--> 子组件
父组件 <--$emit-- 子组件
```

### 4.1 父传子：`props`

父组件在子组件标签上添加自定义属性：

```vue
<!-- App.vue -->
<template>
  <UserInfo :username="username" :age="age" :hobby="hobby" />
</template>

<script>
import UserInfo from './components/UserInfo.vue'
export default {
  components: { UserInfo },
  data () {
    return {
      username: '小帅',
      age: 28,
      hobby: ['篮球', '足球']
    }
  }
}
</script>
```

子组件通过 `props` 接收，在模板中直接使用：

```vue
<!-- UserInfo.vue -->
<template>
  <section>
    <p>姓名：{{ username }}</p>
    <p>年龄：{{ age }}</p>
    <p>爱好：{{ hobby.join('、') }}</p>
  </section>
</template>

<script>
export default {
  props: ['username', 'age', 'hobby']
}
</script>
```

绑定 `:age="age"` 会传递数字；写成 `age="age"` 则会传递字符串字面量 `"age"`。需要传递变量、对象、数组和布尔值时，使用 `v-bind`。

### 4.2 子传父：`$emit`

子组件不能直接修改父组件的数据。它应该触发一个自定义事件，把需要的信息作为参数交给父组件：

```vue
<!-- Son.vue -->
<template>
  <button @click="sendMessage">通知父组件</button>
</template>

<script>
export default {
  methods: {
    sendMessage () {
      this.$emit('change-title', '子组件传来的新标题')
    }
  }
}
</script>
```

父组件监听事件并提供处理函数：

```vue
<!-- App.vue -->
<template>
  <Son @change-title="changeTitle" />
  <h2>{{ title }}</h2>
</template>

<script>
import Son from './components/Son.vue'
export default {
  components: { Son },
  data () { return { title: '原来的标题' } },
  methods: {
    changeTitle (value) {
      this.title = value
    }
  }
}
</script>
```

如果需要传多个值，可以传对象：`this.$emit('save', { id, name })`。事件名建议使用短横线命名，如 `@change-title`。

## 五、Props 及其校验

### 5.1 Props 的特点

- 可以传递任意数量的 prop。
- 可以传递字符串、数字、布尔值、数组、对象和函数等类型。
- Props 是外部传入的数据，子组件只能读取，不能直接修改。

### 5.2 完整校验写法

```js
export default {
  props: {
    width: {
      type: Number,
      required: true,
      validator (value) {
        return value >= 0 && value <= 100
      }
    },
    title: {
      type: String,
      default: '进度'
    },
    options: {
      type: Array,
      default: () => []
    }
  }
}
```

常用配置：

| 配置 | 作用 |
| --- | --- |
| `type` | 类型校验，如 `String`、`Number`、`Boolean`、`Array`、`Object` |
| `required` | 是否必须传入 |
| `default` | 未传入时的默认值 |
| `validator` | 自定义校验，返回 `true` 表示通过 |

`default` 和 `required` 通常不同时使用。数组和对象的默认值必须通过函数返回，避免多个组件实例共享同一个引用。

### 5.3 单向数据流

父级数据通过 props 向下流动，子组件不能直接修改 props：

```js
props: {
  count: Number
},
methods: {
  bad () {
    this.count++ // 不允许：直接修改 prop
  }
}
```

如果子组件需要基于 prop 临时编辑，可以复制一份到自己的 data：

```js
props: { initialCount: Number },
data () {
  return { localCount: this.initialCount }
}
```

如果修改需要影响父组件，使用 `$emit` 通知父组件，由父组件修改自己的数据后重新传入。记忆口诀：**谁的数据谁负责**。

## 六、综合案例：组件版小黑记事本

### 6.1 组件拆分

```text
App.vue
├─ TodoHeader.vue  收集输入、回车或点击添加
├─ TodoMain.vue    接收列表、渲染任务、通知删除
└─ TodoFooter.vue  显示合计、通知清空
```

App 是数据所有者，任务列表放在 App 的 `data` 中。子组件通过 props 获取列表，通过 `$emit` 把“添加、删除、清空”这些意图通知给 App。

### 6.2 App.vue

```vue
<template>
  <div class="todo-app">
    <TodoHeader @add="addTodo" />
    <TodoMain :list="list" @remove="removeTodo" />
    <TodoFooter :list="list" @clear="clearTodos" />
  </div>
</template>

<script>
import TodoHeader from './components/TodoHeader.vue'
import TodoMain from './components/TodoMain.vue'
import TodoFooter from './components/TodoFooter.vue'

export default {
  components: { TodoHeader, TodoMain, TodoFooter },
  data () {
    return {
      list: JSON.parse(localStorage.getItem('todo-list') || '[]')
    }
  },
  watch: {
    list: {
      deep: true,
      handler (value) {
        localStorage.setItem('todo-list', JSON.stringify(value))
      }
    }
  },
  methods: {
    addTodo (name) {
      this.list.unshift({ id: Date.now(), name })
    },
    removeTodo (id) {
      this.list = this.list.filter(item => item.id !== id)
    },
    clearTodos () {
      this.list = []
    }
  }
}
</script>
```

### 6.3 TodoHeader.vue

```vue
<template>
  <header>
    <input v-model.trim="name" @keyup.enter="add" placeholder="请输入任务">
    <button @click="add">添加</button>
  </header>
</template>

<script>
export default {
  data () { return { name: '' } },
  methods: {
    add () {
      if (!this.name) return
      this.$emit('add', this.name)
      this.name = ''
    }
  }
}
</script>
```

### 6.4 TodoMain.vue 与 TodoFooter.vue

```vue
<!-- TodoMain.vue -->
<template>
  <ul>
    <li v-for="item in list" :key="item.id">
      <span>{{ item.name }}</span>
      <button @click="$emit('remove', item.id)">删除</button>
    </li>
  </ul>
</template>
<script>
export default { props: { list: { type: Array, required: true } } }
</script>
```

```vue
<!-- TodoFooter.vue -->
<template>
  <footer v-if="list.length">
    <span>合计：{{ list.length }} 条</span>
    <button @click="$emit('clear')">清空</button>
  </footer>
</template>
<script>
export default { props: { list: { type: Array, required: true } } }
</script>
```

这个案例的关键不是某个标签，而是数据流清楚：列表由 App 维护，子组件只负责展示和发出事件。

## 七、非父子通信：事件总线 EventBus

### 7.1 适用场景

没有直接父子关系的组件之间，可以通过一个共享的 Vue 实例传递简单消息。复杂项目应使用 Vuex、Pinia 或其他状态管理方案。

### 7.2 创建事件总线

```js
// src/utils/EventBus.js
import Vue from 'vue'
export default new Vue()
```

发送方触发事件：

```js
import Bus from '../utils/EventBus'
Bus.$emit('send-msg', '这是 B 组件发送的消息')
```

接收方监听事件：

```js
import Bus from '../utils/EventBus'
export default {
  data () { return { msg: '' } },
  created () {
    this.handleMsg = msg => { this.msg = msg }
    Bus.$on('send-msg', this.handleMsg)
  },
  beforeDestroy () {
    Bus.$off('send-msg', this.handleMsg)
  }
}
```

同一个事件可以被多个组件监听。监听时保存处理函数引用，销毁时用 `$off` 移除，避免组件反复进入页面后重复接收消息。

## 八、非父子通信：`provide` 与 `inject`

### 8.1 作用

祖先组件通过 `provide` 提供数据，后代组件通过 `inject` 注入数据，中间不需要逐层传递 props。

```js
// 祖先组件
export default {
  data () {
    return {
      theme: 'dark',
      userInfo: { name: '小黑' }
    }
  },
  provide () {
    return {
      theme: this.theme,
      userInfo: this.userInfo
    }
  }
}
```

```js
// 任意后代组件
export default {
  inject: ['theme', 'userInfo'],
  created () {
    console.log(this.theme, this.userInfo.name)
  }
}
```

### 8.2 注意事项

- Vue 2 中直接 `provide` 一个简单类型时通常不是响应式的。
- 提供对象时，后代可以观察到对象内部的变化，但不建议让后代随意修改它。
- `provide/inject` 适合主题、用户信息、依赖服务等跨层级共享；业务状态较复杂时使用专门的状态管理。

## 九、`v-model` 原理

在普通文本框上，`v-model` 是 `:value` 和 `@input` 的语法糖：

```vue
<!-- 简写 -->
<input v-model="msg">

<!-- 等价写法 -->
<input :value="msg" @input="msg = $event.target.value">
```

它实现两个方向：

- 数据变化 → 更新输入框的 `value`。
- 输入框变化 → 触发 `input` 事件，更新数据。

`$event` 是模板中接收事件对象的特殊变量。复选框、单选框和下拉框底层使用的属性和事件不同，但使用方式仍然由 Vue 统一处理。

## 十、在组件上使用 `v-model`

### 10.1 默认约定

Vue 2 中，组件上的 `v-model` 默认约定是：

- 子组件用名为 `value` 的 prop 接收值。
- 子组件通过名为 `input` 的事件把新值传回父组件。

父组件：

```vue
<BaseSelect v-model="selectId" />
```

子组件：

```vue
<template>
  <select :value="value" @change="handleChange">
    <option value="101">北京</option>
    <option value="102">上海</option>
    <option value="103">武汉</option>
  </select>
</template>

<script>
export default {
  props: { value: { type: String, required: true } },
  methods: {
    handleChange (event) {
      this.$emit('input', event.target.value)
    }
  }
}
</script>
```

### 10.2 自定义组件的好处

父组件只需要关心 `selectId`，具体的 `<select>` 结构、选项和事件转换封装在子组件中，后续可以替换实现而不改变父组件调用方式。

## 十一、`.sync` 修饰符

`.sync` 适合封装“子组件需要修改父组件某个状态”的场景，比如弹框的显示隐藏。

```vue
<!-- 简写 -->
<BaseDialog :visible.sync="isShow" />

<!-- 完整写法 -->
<BaseDialog
  :visible="isShow"
  @update:visible="isShow = $event"
/>
```

子组件：

```vue
<script>
export default {
  props: { visible: Boolean },
  methods: {
    close () {
      this.$emit('update:visible', false)
    }
  }
}
</script>
```

完整弹框示例：

```vue
<template>
  <div v-show="visible" class="dialog">
    <h3>温馨提示</h3>
    <p>你确认要退出本系统吗？</p>
    <button @click="close">确认</button>
    <button @click="close">取消</button>
  </div>
</template>
<script>
export default {
  props: { visible: Boolean },
  methods: { close () { this.$emit('update:visible', false) } }
}
</script>
```

`.sync` 不是让子组件直接改 prop，而是约定一个 `update:属性名` 事件，真正的数据修改仍由父组件完成。

## 十二、`ref` 与 `$refs`

### 12.1 获取 DOM

```vue
<template>
  <input ref="searchInput">
</template>
<script>
export default {
  mounted () {
    this.$refs.searchInput.focus()
  }
}
</script>
```

### 12.2 获取子组件实例

```vue
<!-- 父组件 -->
<BaseChart ref="chart" />

<script>
export default {
  methods: {
    refresh () {
      this.$refs.chart.render()
    }
  }
}
</script>
```

`$refs` 只在挂载后可用，且只能在当前组件范围内查找。它适合聚焦、测量、调用第三方库和调用子组件暴露的命令式方法；页面显示和状态传递仍然优先使用 props 和事件。

图表组件初始化时，把 DOM 引用交给 ECharts：

```js
mounted () {
  this.chart = echarts.init(this.$refs.chartBox)
  this.chart.setOption({
    xAxis: { data: ['A', 'B', 'C'] },
    yAxis: {},
    series: [{ type: 'bar', data: [5, 20, 36] }]
  })
}
```

## 十三、异步更新与 `$nextTick`

Vue 的数据更新是异步批量处理的：修改响应式数据后，DOM 不会在同一行代码中立刻更新。

错误写法：

```js
editFn () {
  this.isShowEdit = true
  this.$refs.input.focus() // 此时输入框可能还没有被渲染
}
```

正确写法：

```js
editFn () {
  this.isShowEdit = true
  this.$nextTick(() => {
    this.$refs.input.focus()
  })
}
```

`$nextTick` 的回调会在本轮 DOM 更新完成后执行。回调使用箭头函数，保证内部 `this` 仍然指向 Vue 实例。

完整编辑示例：

```vue
<template>
  <div>
    <div v-if="editing">
      <input ref="input" v-model="editValue">
      <button @click="save">确定</button>
    </div>
    <div v-else>
      <span>{{ title }}</span>
      <button @click="startEdit">编辑</button>
    </div>
  </div>
</template>
<script>
export default {
  data () { return { title: '大标题', editing: false, editValue: '' } },
  methods: {
    startEdit () {
      this.editValue = this.title
      this.editing = true
      this.$nextTick(() => this.$refs.input.focus())
    },
    save () {
      this.title = this.editValue.trim() || this.title
      this.editing = false
    }
  }
}
</script>
```

## 十四、常见错误与排查顺序

### 14.1 直接修改 prop

看到“Avoid mutating a prop directly”时，检查子组件是否直接给 prop 赋值。改用 `$emit` 或复制到本地 data。

### 14.2 `data` 写成对象

组件里的 `data` 必须是函数；数组和对象的 prop 默认值也必须使用工厂函数：

```js
props: {
  options: { type: Array, default: () => [] }
}
```

### 14.3 事件总线重复监听

组件每次创建时 `$on`，销毁时要 `$off`。最好保存处理函数引用，而不是只写匿名函数。

### 14.4 `$refs` 为空

确认元素已经渲染，并把访问代码放在 `mounted` 或 `$nextTick` 中。如果元素受 `v-if` 控制，修改条件后必须等待下一次 DOM 更新。

### 14.5 `.sync` 事件名写错

`visible.sync` 对应的事件名必须是 `update:visible`，大小写和属性名都要一致。

### 14.6 样式仍然冲突

检查是否忘记 `scoped`，或者样式本身放在了全局入口文件。必要时给组件根元素增加唯一类名。

## 十五、练习题

1. 创建 `BaseProgress` 组件，使用 `props` 接收 `width`，限制范围为 0 到 100，并通过 `validator` 校验。
2. 创建 `BaseSelect` 组件，用 `v-model` 实现父组件和下拉框的双向绑定。
3. 创建 `BaseDialog`，使用 `.sync` 控制显示隐藏，点击遮罩和关闭按钮都能通知父组件关闭。
4. 将小黑记事本拆成头部、列表、底部三个组件，并使用 `localStorage` 持久化。
5. 创建两个没有父子关系的组件，用 EventBus 发送消息；切换页面后确认监听已清理。
6. 创建一个图表组件，使用 `ref` 获取容器，在 `mounted` 中初始化，在销毁时释放图表实例。

## 十六、当天速记

- 组件样式默认可能全局生效，局部样式使用 `<style scoped>`。
- 组件 `data` 必须是函数，每个实例才能拥有独立状态。
- 父传子使用 `props`，子传父使用 `$emit`；数据由所有者负责修改。
- Props 可以做类型、必填、默认值和自定义校验。
- 非父子通信可用 EventBus；跨层级共享可用 `provide/inject`。
- Vue 2 组件 `v-model` 默认对应 `value` prop 和 `input` 事件。
- `.sync` 对应 `:prop` 与 `@update:prop`。
- `$refs` 用于访问当前组件的 DOM 或子组件实例。
- 修改数据后立即操作新 DOM，要使用 `$nextTick`。